# Open Addressing<br/>Linear and Quadratic Probing


## TL/DR

Write code to measure frequency of collisions as a function of $\alpha$.

_Hint 1:_ Counting probes is a perfectly acceptable proxy for frequency of collisions.<br/>_Hint 2:_ you may want to read the rest of this notebook.


## Collisions are inevitable

**Open addressing** is a collision resolution strategy.

A **collision** occurs when two distinct entries produce the same index after being passed through the hash function. For example, given the position function below:


In [ ]:
def position(entry: str, m: int) -> int:
    return hash(entry) % m

strings `Alice` and `Charlie` hash to the same slot of the underlying array when `m=4`. If we use function `position` above to store these strings in a list of size `m`, we'll have a collision.

Only one entry can occupy a given slot, of course, so we need a strategy to address collision.

One way to resolve the collision is **chaining.** It uses linked lists to hold multiple entries, at the same array (list) position, as shown below.

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/data_structures/chaining.png)

Chaining is demonstrated in [class `SimpleHashTable`](./live_coding.ipynb).


**Probing** is another way to resolve collisions. Using the example above, if `Alice` gets to position `[2]` first, and the position next to it is open, then `Charlie` could go there.

![](https://raw.githubusercontent.com/lgreco/images/refs/heads/main/data_structures/probing.png)

In this case all entries are stored directly in the array itself — there are no external chains of linked lists. When a collision occurs (two entries hash to the same index), we search for the next available slot within the array by following a deterministic **probe sequence**. The general form is:

```python
probe(entry, i) = (hash(entry) + f(i)) % m
```

where `m` is the table capacity, `i` is the attempt number (0, 1, 2, …), and `f(i)` defines the probing strategy (e.g., `f(i) = i` for linear, `f(i) = i*i` for quadratic).


## Load factor

The load factor of a hash table is defined as

$$
\alpha = \frac{\text{elements of underlying array used}}{\text{length of underlying array}}
$$

In the purple example above, $\alpha = 1/4$ and in the orange example $\alpha = 2/4$. Obviously the load factor $\alpha$ can never exceed 1. At $\alpha=1$ every slot of the array is occupied and it must be resized. In practice, performance degrades well before that point, so implementations typically resize the underlying array as soon as $\alpha$ exceeds a threshold.


## General Probing Framework

Given a table of capacity `m` and a hash function `hash(entry)`, the probe sequence for a key is:

```python
probe(entry, i) = (hash(entry) + f(i)) % m  #  for i = 0, 1, 2, ...
```

where `f(i)` is the **probe function** and `i` is the attempt number (starting at 0). The two schemes differ only in their choice of `f(i)`.

| Scheme    | $f(i)$       |
| --------- | ------------ |
| Linear    | $f(i) = i$   |
| Quadratic | $f(i) = i^2$ |


## Linear Probing

On a collision at index `position`, we simply check the next slot, then the next, and so on, _wrapping_ around the array.

### Primary Clustering

Linear probing's main weakness is **primary clustering**. Once a contiguous block of occupied slots forms, any new entry that hashes into that block must traverse the entire cluster before finding an empty slot. Worse, doing so _extends_ the cluster, making future collisions even more expensive. As load factor increases, clusters merge and probe lengths grow rapidly.

### Properties

- Cache-friendly: probes visit consecutive memory locations.
- Simple to implement.
- Performance degrades sharply as the load factor $\alpha$ approaches 1.
- Average successful search: approximately $(1+1/(1-\alpha))/2$.
- Average unsuccessful search: approximately $(1 + (1/(1 − \alpha))^2)/2$.


## Quadratic Probing

On a collision, the probe jumps by 1, then 4, then 9, then 16, etc. The increasing step sizes help probes "escape" clusters more quickly.

### Secondary Clustering

Quadratic probing eliminates primary clustering but introduces **secondary clustering**: all keys that hash to the _same_ index follow the _same_ quadratic probe sequence. This is less severe than primary clustering but still causes extra work when many keys share a hash value.

### The Coverage Guarantee

Quadratic probing is **not guaranteed to visit every slot**. However, there is an important theorem:

> If $m$ is prime and the table is less than half full ($\alpha < 0.5$), then quadratic probing will always find an empty slot.

This is why implementations using quadratic probing typically choose a prime table size and resize when the load factor exceeds 0.5.

### Properties

- Reduces primary clustering.
- Slightly less cache-friendly than linear probing (probes are not contiguous).
- Requires care with table size (prime) and load factor (< 0.5 for the guarantee).
- Subject to secondary clustering.


## Side-by-Side Comparison

| Property             | Linear Probing       | Quadratic Probing         |
| -------------------- | -------------------- | ------------------------- |
| Probe function       | $f(i) = i$           | $f(i) = i^2$              |
| Primary clustering   | Yes (main weakness)  | No                        |
| Secondary clustering | No                   | Yes                       |
| Cache performance    | Excellent            | Good                      |
| Guaranteed insertion | Always (if not full) | Only if m is prime, α<0.5 |
| Typical max load     | ~0.7 practical limit | ~0.5 for the guarantee    |
| Implementation       | Simplest             | Slightly more complex     |


## Deletion: The Tombstone Problem

With open addressing, you cannot simply mark a slot as empty when deleting. Doing so would break the probe chain for keys that were inserted _past_ the deleted slot. The standard solution is **lazy deletion** using a **tombstone** (a special sentinel value):

- On delete: replace the slot with a tombstone marker.
- On search: treat tombstones as occupied (keep probing past them).
- On insert: treat tombstones as available (reuse the slot).

Over time, tombstones accumulate and degrade performance. A periodic rehash into a fresh table cleans them out.

## Key Takeaways

1. Both schemes store all entries directly in the array with no auxiliary data structures.
2. They differ in a single line of code: the probe function $f(i)$.
3. Linear probing is simple and cache-friendly but suffers from primary clustering.
4. Quadratic probing mitigates clustering but requires a prime table size and a lower load factor for its insertion guarantee.
5. Neither scheme handles deletion cleanly — tombstones are required.
6. In practice, keeping the load factor well below the theoretical limits (resize and rehash when needed) is essential for both schemes.


In [ ]:
from __future__ import annotations


class ProbingHashTable:
    """Hash table demonstrating linear vs. quadratic probing."""

    _DEFAULT_CAPACITY = 11
    _PROBE_MODES = ("linear", "quadratic")
    _DEFAULT_PROBING_MODE = _PROBE_MODES[0]
    _ERROR_MODE = f"mode must be one of {', '.join(_PROBE_MODES)}"
    _ERROR_FULL = "Hash table is full"

    def __init__(
        self, capacity: int = _DEFAULT_CAPACITY, mode: str = _DEFAULT_PROBING_MODE
    ):
        if mode not in self._PROBE_MODES:
            raise ValueError(self._ERROR_MODE)
        self._capacity = capacity
        self._mode = mode
        # Each slot is a list: [key, value] when occupied, or None
        self._table: list[list | None] = [None] * capacity
        self._size = 0

    def _hash(self, key: int) -> int:
        return key % self._capacity

    def _probe(self, position: int, attempt: int) -> int:
        if self._mode == self._DEFAULT_PROBING_MODE:  # linear
            return (position + attempt) % self._capacity
        else:  # quadratic
            return (position + attempt * attempt) % self._capacity

    def insert(self, value: str) -> list[int]:
        """Insert key-value pair. Returns list of indices probed."""
        # Initialize list to track probe indices
        probes = []
        # Check if the hash table has space for a new entry.
        # If not, we will not attempt to insert and will return
        # an empty list of probes.
        if self._size < self._capacity:
            # Generate a non-negative integer key from the value
            key: int = abs(hash(value))
            # Convert the key to an index in the underlying table
            index = self._hash(key)
            # Attempt to insert the key-value pair, probing for an empty
            # slot.
            i: int = 0
            current_size: int = self._size
            # Loop to attempt to find an empty slot. The loop ends as soon as
            # we have probed the entire table or successfully inserted the
            # value.
            while i < self._capacity and current_size == self._size:
                # Location in the underlying table to check
                probe_index = self._probe(index, i)
                # Record the location we are probing
                probes.append(probe_index)
                # Obtain the contents at the location we are probing
                slot = self._table[probe_index]
                if slot is None:
                    # The slot is empty, so we can insert the key-value pair here
                    self._table[probe_index] = value
                    # Update the size of the hash table to reflect the new insertion.
                    # This will also cause the loop to end since current_size will
                    # no longer equal self._size
                    self._size += 1
                # If the slot is not empty, we will try the next probe index
                # in the next iteration of the loop.
                i += 1
        return probes

    def contains(self, value: str) -> bool:
        """Check if value is in the hash table."""
        key = abs(hash(value))
        index = self._hash(key)
        found: bool = False
        i: int = 0
        while i < self._capacity and not found:
            probe_index = self._probe(index, i)
            slot = self._table[probe_index]
            found = slot == value
            i += 1
        return found

    def display(self) -> str:
        lines = [f"{'Idx':<5}{'Content'}"]
        lines.append("-" * 20)
        for i, slot in enumerate(self._table):
            content = f"{slot[0]} -> {slot[1]}" if slot else "---"
            lines.append(f"{i:<5}{content}")
        return "\n".join(lines)


# ── Demo ──────────────────────────────────────────────────────


def demo():
    keys = [10, 22, 31, 4, 15, 28, 17, 88, 59]

    for mode in ("linear", "quadratic"):
        ht = ProbingHashTable(capacity=11, mode=mode)
        print(f"\n{'=' * 40}")
        print(f"  {mode.upper()} PROBING (capacity=11)")
        print(f"{'=' * 40}")

        for key in keys:
            probes = ht.insert(key, f"v{key}")
            status = "collision!" if len(probes) > 1 else "direct"
            print(f"  insert({key:>2}): hash={key % 11:<3} probes={probes}  [{status}]")

        print(f"\n{ht.display()}")
        print(
            f"\nLoad factor: {ht._size}/{ht._capacity} = {ht._size / ht._capacity:.2f}"
        )


if __name__ == "__main__":
    demo()


  LINEAR PROBING (capacity=11)
  insert(10): hash=10  probes=[10]  [direct]
  insert(22): hash=0   probes=[0]  [direct]
  insert(31): hash=9   probes=[9]  [direct]
  insert( 4): hash=4   probes=[4]  [direct]
  insert(15): hash=4   probes=[4, 5]  [collision!]
  insert(28): hash=6   probes=[6]  [direct]
  insert(17): hash=6   probes=[6, 7]  [collision!]
  insert(88): hash=0   probes=[0, 1]  [collision!]
  insert(59): hash=4   probes=[4, 5, 6, 7, 8]  [collision!]

Idx  Content
--------------------
0    22 -> v22
1    88 -> v88
2    ---
3    ---
4    4 -> v4
5    15 -> v15
6    28 -> v28
7    17 -> v17
8    59 -> v59
9    31 -> v31
10   10 -> v10

Load factor: 9/11 = 0.82

  QUADRATIC PROBING (capacity=11)
  insert(10): hash=10  probes=[10]  [direct]
  insert(22): hash=0   probes=[0]  [direct]
  insert(31): hash=9   probes=[9]  [direct]
  insert( 4): hash=4   probes=[4]  [direct]
  insert(15): hash=4   probes=[4, 5]  [collision!]
  insert(28): hash=6   probes=[6]  [direct]
  insert(17): h